# Stage 1 — Candidate Expert Selection

This notebook visualises which experts were selected as candidates for Stage 2 intervention.

**Selection logic**: for each layer, take the top-N experts by |RD| from both the frequency-based and logit-based RD metrics, then take only their **intersection**. This ensures selected experts show a consistent signal on both metrics — reducing noise.

Candidates are selected separately for:
- **Safety (negative direction)**: suppress compliance-preferred experts → expect safer outputs
- **Safety (positive direction)**: suppress refusal-preferred experts → expect less safe outputs
- **Faithfulness (negative direction)**: suppress confabulation-preferred experts → expect more faithful outputs

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sys.path.insert(0, os.path.join('..', 'src'))
sys.path.insert(0, os.path.join('..', '..', 'stage2', 'src'))
from rd_utils import load_rd
from candidates import select_candidates

sns.set_theme(style='whitegrid', font='serif')
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 150,
})

RD_FAITH_FREQ   = '/scratch/sc23jc3/results/rd_faithfulness.json'
RD_FAITH_LOGIT  = '/scratch/sc23jc3/results/rd_faithfulness_logits.json'
RD_SAFETY_FREQ  = '/scratch/sc23jc3/results/rd_safety.json'
RD_SAFETY_LOGIT = '/scratch/sc23jc3/results/rd_safety_logits.json'
CANDIDATE_N = 10

OUT_DIR = 'figures'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
safety_neg  = select_candidates(RD_SAFETY_FREQ,  RD_SAFETY_LOGIT,  CANDIDATE_N, direction='negative')
safety_pos  = select_candidates(RD_SAFETY_FREQ,  RD_SAFETY_LOGIT,  CANDIDATE_N, direction='positive')
faith_neg   = select_candidates(RD_FAITH_FREQ,   RD_FAITH_LOGIT,   CANDIDATE_N, direction='negative')

print(f'Safety negative candidates: {sum(len(v) for v in safety_neg.values())} experts across {len(safety_neg)} layers')
print(f'Safety positive candidates: {sum(len(v) for v in safety_pos.values())} experts across {len(safety_pos)} layers')
print(f'Faith negative candidates:  {sum(len(v) for v in faith_neg.values())} experts across {len(faith_neg)} layers')

## 1. Candidate Count per Layer

How many experts were selected per layer for each candidate set?

In [ ]:
rd_safety = load_rd(RD_SAFETY_FREQ)
all_layers = sorted(rd_safety.keys(), key=lambda x: int(x.split('.')[2]))
layer_indices = [int(l.split('.')[2]) for l in all_layers]

def counts_per_layer(candidates, all_layers):
    return [len(candidates.get(l, [])) for l in all_layers]

sn = counts_per_layer(safety_neg, all_layers)
sp = counts_per_layer(safety_pos, all_layers)
fn = counts_per_layer(faith_neg, all_layers)

x = np.array(layer_indices)
w = 0.25
fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(x - w, sn, width=w, label='Safety negative', color='#2980b9', alpha=0.85)
ax.bar(x,     sp, width=w, label='Safety positive', color='#e74c3c', alpha=0.85)
ax.bar(x + w, fn, width=w, label='Faith negative',  color='#27ae60', alpha=0.85)
ax.set_xlabel('Layer Index')
ax.set_ylabel('Number of Selected Experts')
ax.set_title(f'Candidate Experts per Layer (N={CANDIDATE_N})')
ax.legend()
ax.set_xticks(x[::2])
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'candidate_counts.png'), dpi=300, bbox_inches='tight')
plt.show()

## 2. Candidate Map — Safety Negative

Which specific (layer, expert) pairs were selected?

In [ ]:
def plot_candidate_map(candidates, all_layers, rd_by_layer, title, filename, color):
    n_layers  = len(all_layers)
    n_experts = len(next(iter(rd_by_layer.values())))
    matrix = np.zeros((n_layers, n_experts))
    for i, layer in enumerate(all_layers):
        if layer in candidates:
            for expert_idx in candidates[layer]:
                matrix[i, expert_idx] = 1

    layer_labels = [int(l.split('.')[2]) for l in all_layers]
    fig, ax = plt.subplots(figsize=(18, 7))
    ax.imshow(matrix, aspect='auto', cmap=plt.cm.colors.ListedColormap(['#f0f0f0', color]), vmin=0, vmax=1)
    ax.set_xlabel('Expert Index')
    ax.set_ylabel('Layer Index')
    ax.set_title(title)
    ax.set_yticks(range(n_layers))
    ax.set_yticklabels(layer_labels, fontsize=8)
    ax.set_xticks(range(0, n_experts, 8))
    selected_patch = mpatches.Patch(color=color, label='Selected')
    ax.legend(handles=[selected_patch], loc='upper right')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, filename), dpi=300, bbox_inches='tight')
    plt.show()

plot_candidate_map(
    safety_neg, all_layers, rd_safety,
    'Selected Candidates — Safety Negative Direction (suppress unsafe experts)',
    'candidates_safety_neg.png', '#2980b9'
)

## 3. Candidate Map — Safety Positive

In [ ]:
plot_candidate_map(
    safety_pos, all_layers, rd_safety,
    'Selected Candidates — Safety Positive Direction (suppress safe experts)',
    'candidates_safety_pos.png', '#e74c3c'
)

## 4. Candidate Map — Faithfulness Negative

In [ ]:
rd_faith = load_rd(RD_FAITH_FREQ)
faith_layers = sorted(rd_faith.keys(), key=lambda x: int(x.split('.')[2]))
plot_candidate_map(
    faith_neg, faith_layers, rd_faith,
    'Selected Candidates — Faithfulness Negative Direction (suppress confabulation experts)',
    'candidates_faith_neg.png', '#27ae60'
)

## Discussion

*(Fill in after examining the plots.)*

- Are candidates distributed throughout the network or concentrated in specific layers?
- Is the intersection (freq ∩ logit) sparse or dense — does the filtering remove many candidates?
- Do safety and faithfulness candidates overlap at all? Overlap would suggest shared expert circuits.
- Does CANDIDATE_N=10 seem appropriate, or should it be adjusted based on the natural distribution of |RD| values?